# Drivetrain Feed-forward Calibration
Using logs containing measurements from the motor encoders, determine calibration settings for drivetrain motors.

This was built for a Romi, but might be useful to reference for other drivetrain calibration efforts.

## Setup
Cells in this section are used to setup the notebook

In [ ]:
from robot_calibration.logger import get_logger


logger = get_logger("drivetrain_feed_forward_calibration")

## Data Loading
Cells in this section are used to load and transform log data for use in calibration

In [ ]:
from robot_calibration.drivetrain_feed_forward import read_logs


log_file = "/Users/cory.huebner/workspace/robo-raiders/robot-calibration/.data/FRC_20260930_065500.csv"
fitting_data = read_logs(log_file, max_time=350)
fitting_data

In [ ]:
# Visualize the voltage impact on motors

from robot_calibration.drivetrain_feed_forward import (
    get_chart_of_voltage_impact_on_motors,
)


chart = get_chart_of_voltage_impact_on_motors(fitting_data, "Voltage Impact on Motors (Raw Encoder Data)")
chart.show()


In [ ]:
# Visualize the voltage impact on motors with smoothed encoder data

from robot_calibration.drivetrain_feed_forward import (
    get_chart_of_voltage_impact_on_motors,
    remove_noise_from_motor_rates,
)


fitting_data = remove_noise_from_motor_rates(fitting_data)
chart = get_chart_of_voltage_impact_on_motors(fitting_data, "Voltage Impact on Motors (Raw Encoder Data)")
chart.show()


In [ ]:
# Fit the left motor
from robot_calibration.drivetrain_feed_forward import fit_feedforward, get_fitted_voltage_predictions


left_motor_fitting_results = fit_feedforward(fitting_data["voltage"], fitting_data["left_motor_rate"])
logger.info(f"Left motor fitting results: kS={left_motor_fitting_results.kS}, kV={left_motor_fitting_results.kV}")
fitting_data["left_motor_predicted_voltage"] = get_fitted_voltage_predictions(
    fitting_data["left_motor_rate"], left_motor_fitting_results.kS, left_motor_fitting_results.kV
)


In [ ]:
# Plot predicted voltages to see how well the feed-forward model fit the left motor
from robot_calibration.drivetrain_feed_forward import get_chart_of_actual_vs_predicted_voltage


get_chart_of_actual_vs_predicted_voltage(fitting_data, "left_motor_rate", "Left Motor Actual vs. Predicted Voltages")


In [ ]:
# Fit the right motor
from robot_calibration.drivetrain_feed_forward import fit_feedforward, get_fitted_voltage_predictions


right_motor_fitting_results = fit_feedforward(fitting_data["voltage"], fitting_data["right_motor_rate"])
logger.info(f"Right motor fitting results: kS={right_motor_fitting_results.kS}, kV={right_motor_fitting_results.kV}")
fitting_data["right_motor_predicted_voltage"] = get_fitted_voltage_predictions(
    fitting_data["right_motor_rate"], right_motor_fitting_results.kS, right_motor_fitting_results.kV
)


In [ ]:
# Plot predicted voltages to see how well the feed-forward model fit the right motor
from robot_calibration.drivetrain_feed_forward import get_chart_of_actual_vs_predicted_voltage


get_chart_of_actual_vs_predicted_voltage(fitting_data, "left_motor_rate", "Left Motor Actual vs. Predicted Voltages")

In [ ]:
# Print the left vs. right motor feed-forward constants for comparison sake
from pandas import DataFrame


DataFrame(
    [
        {
            "leftMotorKS": left_motor_fitting_results.kS,
            "leftMotorKV": left_motor_fitting_results.kV,
            "rightMotorKS": right_motor_fitting_results.kS,
            "rightMotorKV": right_motor_fitting_results.kV,
        }
    ]
)